# Feirando - Step 11: Python analysis
Adds what SQL can't do well: significance tests (H1, H5), distance bands and raw delivery time (H4), the seller vs carrier split (H3), and an export for Tableau.

In [4]:
import duckdb
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import ttest_ind
from statsmodels.stats.proportion import proportions_ztest

CLEAN = Path.home() / "Feirando" / "clean"
OUT = Path.home() / "Feirando" / "analysis"
OUT.mkdir(exist_ok=True)

con = duckdb.connect()
zip_types = {
    "customers":   {"customer_zip_code_prefix": "VARCHAR"},
    "sellers":     {"seller_zip_code_prefix": "VARCHAR"},
    "geolocation": {"geolocation_zip_code_prefix": "VARCHAR"},
}
tables = ["orders", "order_items", "order_reviews", "order_payments", "customers",
          "geolocation", "products", "sellers", "category_translation"]

for t in tables:
    path = CLEAN / f"{t}_clean.csv"
    opts = f", types={zip_types[t]}" if t in zip_types else ""
    con.execute(f"CREATE OR REPLACE TABLE {t} AS SELECT * FROM read_csv('{path}'{opts})")

results = []
con.sql("SELECT table_name, estimated_size AS rows FROM duckdb_tables() ORDER BY table_name")

┌──────────────────────┬────────┐
│      table_name      │  rows  │
│       varchar        │ int64  │
├──────────────────────┼────────┤
│ category_translation │     74 │
│ customers            │  99441 │
│ geolocation          │  19010 │
│ order_items          │ 112650 │
│ order_payments       │ 103886 │
│ order_reviews        │  98673 │
│ orders               │  99441 │
│ products             │  32951 │
│ sellers              │   3095 │
└──────────────────────┴────────┘

## 1. H1 : is the review gap real?
Welch's t-test compares two averages (late vs on-time review scores). p < 0.05 = significant. If p prints as 0, report it as p < 0.001.

In [5]:
scores = con.sql("""
    SELECT o.is_late, r.review_score
    FROM orders o
    JOIN order_reviews r ON o.order_id = r.order_id
    WHERE o.in_delivery_scope AND o.in_window
""").df()

late    = scores.loc[scores["is_late"], "review_score"]
on_time = scores.loc[~scores["is_late"], "review_score"]

t, p = ttest_ind(late, on_time, equal_var=False)
gap = on_time.mean() - late.mean()
print(f"Late {late.mean():.2f} | On time {on_time.mean():.2f} | Gap {gap:.2f} | t = {t:.1f} | p = {p:.3g}")

results.append({"test": "H1 review gap", "result": f"{gap:.2f} stars", "threshold": "0.2 stars",
                "p_value": "< 0.001" if p < 0.001 else f"{p:.3f}"})

Late 2.27 | On time 4.29 | Gap 2.02 | t = -100.8 | p = 0


## 2. H5 - is the repeat-purchase gap real?
Rebuilds the H5 tables from the SQL notebook, then a two-proportion z-test compares the two repeat rates.

In [6]:
con.sql("""
    CREATE OR REPLACE TABLE customer_orders AS
    SELECT o.order_id, o.is_late, c.customer_unique_id, o.order_purchase_timestamp
    FROM orders o JOIN customers c ON o.customer_id = c.customer_id
""")
con.sql("""
    CREATE OR REPLACE TABLE first_dates AS
    SELECT customer_unique_id, MIN(order_purchase_timestamp) AS first_order
    FROM customer_orders GROUP BY customer_unique_id
""")
con.sql("""
    CREATE OR REPLACE TABLE first_orders AS
    SELECT co.* FROM customer_orders co
    JOIN first_dates fd ON co.customer_unique_id = fd.customer_unique_id
                       AND co.order_purchase_timestamp = fd.first_order
""")
con.sql("""
    CREATE OR REPLACE TABLE h5_shoppers AS
    SELECT fo.* FROM first_orders fo
    JOIN (SELECT customer_unique_id FROM first_orders
          GROUP BY customer_unique_id HAVING COUNT(*) = 1) single USING (customer_unique_id)
    WHERE fo.is_late IS NOT NULL
      AND fo.order_purchase_timestamp >= '2017-01-01'
      AND fo.order_purchase_timestamp <  '2018-03-01'
""")
con.sql("""
    CREATE OR REPLACE TABLE h5_result AS
    SELECT h.customer_unique_id, h.is_late,
           EXISTS (SELECT 1 FROM customer_orders co
                   WHERE co.customer_unique_id = h.customer_unique_id
                     AND co.order_purchase_timestamp >  h.order_purchase_timestamp
                     AND co.order_purchase_timestamp <= h.order_purchase_timestamp + INTERVAL 180 DAY) AS repeated
    FROM h5_shoppers h
""")

h5 = con.sql("""
    SELECT is_late AS first_order_late, COUNT(*) AS shoppers,
           SUM(CASE WHEN repeated THEN 1 ELSE 0 END) AS repeaters
    FROM h5_result GROUP BY is_late ORDER BY is_late DESC
""").df()
h5["repeaters"] = h5["repeaters"].astype(int)
h5["repeat_rate_pct"] = (h5["repeaters"] / h5["shoppers"] * 100).round(2)

z, p = proportions_ztest(h5["repeaters"], h5["shoppers"])
gap_pp = h5["repeat_rate_pct"].iloc[1] - h5["repeat_rate_pct"].iloc[0]
print(h5.to_string(index=False))
print(f"Gap {gap_pp:.2f} pp | z = {z:.2f} | p = {p:.3f}")

results.append({"test": "H5 repeat gap", "result": f"{gap_pp:.2f} pp", "threshold": "0.5 pp", "p_value": f"{p:.3f}"})

 first_order_late  shoppers  repeaters  repeat_rate_pct
             True      3655         89             2.44
            False     51350       1475             2.87
Gap 0.43 pp | z = -1.54 | p = 0.124


## 3. H4 - does lateness grow with distance?
Distance in km between seller and customer zip pins (haversine formula, because the Earth is round). Per band: late rate, median delivery days (raw speed) and median promised days (the buffer). Orders with a missing pin (~0.3%) drop out of the bands only.

In [7]:
h4 = con.sql("""
    SELECT DISTINCT
           o.order_id, o.is_late,
           date_diff('day', CAST(o.order_purchase_timestamp AS DATE), CAST(o.order_delivered_customer_date AS DATE)) AS delivery_days,
           date_diff('day', CAST(o.order_purchase_timestamp AS DATE), CAST(o.order_estimated_delivery_date AS DATE)) AS promised_days,
           DATE_TRUNC('month', o.order_purchase_timestamp) AS purchase_month,
           c.customer_state, s.seller_state,
           gc.lat AS cust_lat, gc.lng AS cust_lng, gs.lat AS sell_lat, gs.lng AS sell_lng
    FROM orders o
    JOIN customers c    ON o.customer_id = c.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN sellers s      ON oi.seller_id = s.seller_id
    LEFT JOIN geolocation gc ON c.customer_zip_code_prefix = gc.geolocation_zip_code_prefix
    LEFT JOIN geolocation gs ON s.seller_zip_code_prefix  = gs.geolocation_zip_code_prefix
    WHERE o.in_delivery_scope AND o.in_window AND NOT o.multi_seller
""").df()
print(len(h4), h4["order_id"].is_unique)   # expect 94931 True

def haversine_km(lat1, lng1, lat2, lng2):
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

h4["distance_km"] = haversine_km(h4["sell_lat"], h4["sell_lng"], h4["cust_lat"], h4["cust_lng"])
h4["state_match"] = np.where(h4["customer_state"] == h4["seller_state"], "same_state", "cross_state")
h4["distance_band"] = pd.cut(h4["distance_km"], bins=[0, 100, 500, 1000, 2000, np.inf],
                             labels=["<100 km", "100-500 km", "500-1,000 km", "1,000-2,000 km", "2,000+ km"],
                             include_lowest=True)

by_band = (h4.groupby("distance_band", observed=True)
             .agg(orders=("order_id", "count"),
                  late_rate_pct=("is_late", lambda s: round(s.mean() * 100, 1)),
                  median_delivery_days=("delivery_days", "median"),
                  median_promised_days=("promised_days", "median"))
             .reset_index())
by_band

94931 True


,distance_band,orders,late_rate_pct,median_delivery_days,median_promised_days
0,<100 km,17651,4.5,5.0,15.0
1,100-500 km,36352,6.2,9.0,23.0
2,"500-1,000 km",25322,7.2,12.0,26.0
3,"1,000-2,000 km",9585,9.6,15.0,30.0
4,"2,000+ km",5550,12.1,18.0,34.0


## 4. H3 - seller delay or carrier delay?
Carrier date after the handover deadline → the seller was late. Carrier date on time but the order still arrived late → the delay was with the carrier. Single-seller orders with both dates known; for several items, the latest deadline is used.

In [8]:
h3 = con.sql("""
    SELECT o.order_id, o.is_late,
           o.order_delivered_carrier_date AS carrier_date,
           MAX(oi.shipping_limit_date)    AS handover_deadline
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.in_delivery_scope AND o.in_window AND NOT o.multi_seller
    GROUP BY o.order_id, o.is_late, o.order_delivered_carrier_date
""").df()

h3 = h3.dropna(subset=["carrier_date", "handover_deadline"]).copy()
h3["seller_late_handover"] = h3["carrier_date"] > h3["handover_deadline"]

seller_share = h3.loc[h3["is_late"], "seller_late_handover"].mean() * 100
print(f"Late orders where the seller handed over late: {seller_share:.1f}%")
print(f"Late orders where the seller was on time (carrier side): {100 - seller_share:.1f}%")

results.append({"test": "H3 late orders with late seller handover", "result": f"{seller_share:.1f}%",
                "threshold": "", "p_value": ""})

h3.groupby("seller_late_handover").agg(
    orders=("order_id", "count"),
    late_rate_pct=("is_late", lambda s: round(s.mean() * 100, 1))).reset_index()

Late orders where the seller handed over late: 27.8%
Late orders where the seller was on time (carrier side): 72.2%


,seller_late_handover,orders,late_rate_pct
0,False,86399,5.4
1,True,8530,21.2


In [9]:
extra = con.sql("""
    SELECT o.order_id, o.delay_days, r.review_score
    FROM orders o
    LEFT JOIN order_reviews r ON o.order_id = r.order_id
    WHERE o.in_delivery_scope AND o.in_window AND NOT o.multi_seller
""").df()

tableau = (h4.drop(columns=["cust_lat", "cust_lng", "sell_lat", "sell_lng"])
             .merge(extra, on="order_id", how="left")
             .merge(h3[["order_id", "seller_late_handover"]], on="order_id", how="left"))
tableau["delay_band"] = pd.cut(tableau["delay_days"], bins=[-np.inf, 0, 3, 10, np.inf],
                               labels=["On time", "1-3 days late", "4-10 days late", "11+ days late"])
tableau["distance_km"] = tableau["distance_km"].round(0)

tableau.to_csv(OUT / "tableau_orders.csv", index=False)
print(tableau.shape)   # expect about (94931, 16)

(94931, 14)


## 5. Summary

In [10]:
pd.DataFrame(results)

,test,result,threshold,p_value
0,H1 review gap,2.02 stars,0.2 stars,< 0.001
1,H5 repeat gap,0.43 pp,0.5 pp,0.124
2,H3 late orders with late seller handover,27.8%,,


## Lets check if the late deliveres are scored only because they are delayed or because the products aare bad and end up getting low review. Then the review score also depends on the product quality ont only beacuase it was delivered late. 

In [13]:
ox = con.sql("""
    SELECT o.order_id, o.is_late, r.review_score, c.customer_state,
           SUM(oi.price) AS order_value
    FROM orders o
    JOIN customers c    ON o.customer_id = c.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN order_reviews r ON o.order_id = r.order_id
    WHERE o.in_delivery_scope AND o.in_window
    GROUP BY o.order_id, o.is_late, r.review_score, c.customer_state
""").df()

ox["price_group"] = pd.qcut(ox["order_value"], 3, labels=["Cheap", "Mid", "Expensive"])

check1 = ox.pivot_table(index="price_group", columns="is_late", values="review_score",
                        aggfunc="mean", observed=True).round(2)
check1.columns = ["on_time", "late"]
check1["gap"] = check1["on_time"] - check1["late"]
check1



,on_time,late,gap
price_group,,,
Cheap,4.33,2.32,2.01
Mid,4.31,2.31,2.00
Expensive,4.23,2.20,2.03


## Late rate by customer state. Which regions drive late deliveries?

In [14]:
check2 = (ox.groupby("customer_state")
            .agg(orders=("order_id", "count"),
                 late_rate_pct=("is_late", lambda s: round(s.mean() * 100, 1)))
            .query("orders >= 500")
            .sort_values("late_rate_pct", ascending=False))
check2.head(5)

,orders,late_rate_pct
customer_state,,
MA,709,17.2
CE,1267,13.7
RJ,12172,12.0
BA,3226,11.9
PA,929,10.8
